# Product Launch Risk Assessment | Mixture-of-Agents (MoA)

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
from concurrent.futures import ThreadPoolExecutor
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
# Three distinct personas enforce genuinely different reasoning
# NOTE: In production, use genuinely different model providers for diversity:
#   proposer_1 = ChatOpenAI(model="gpt-4o")       # OpenAI
#   proposer_2 = ChatAnthropic(model="claude-sonnet-4-5-20250929")  # Anthropic
#   proposer_3 = ChatGoogleGenerativeAI(model="gemini-2.0-flash")   # Google
# This demo uses persona-driven diversity with one provider as a portable example.

PROPOSERS = {
    "optimist": (
        "You are an optimistic strategist. Focus ONLY on opportunities, market upside, "
        "competitive advantages, and best-case outcomes. Argue why this will succeed."
    ),
    "skeptic": (
        "You are a risk-focused analyst. Focus ONLY on threats, failure modes, regulatory "
        "dangers, and worst-case scenarios. Be brutally honest about what could go wrong."
    ),
    "pragmatist": (
        "You are an execution-focused ops lead. Focus ONLY on practical execution: "
        "timelines, resource needs, dependencies, and bottlenecks. Ignore big-picture "
        "strategy — answer 'how do we actually ship this?'"
    ),
}

aggregator = ChatOpenAI(model="gpt-4o", temperature=0)

class MoAState(TypedDict):
    question: str
    proposals: List[str]
    final_answer: str

In [4]:
def generate_proposals(state: MoAState) -> dict:
    """Run the same prompt through 3 persona-driven models in parallel."""
    def get_proposal(item: tuple) -> str:
        name, system_prompt = item
        llm = ChatOpenAI(model="gpt-4o", temperature=0.4)
        response = llm.invoke([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": state["question"]},
        ])
        return f"[{name.upper()}]: {response.content}"

    with ThreadPoolExecutor(max_workers=3) as executor:
        proposals = list(executor.map(get_proposal, PROPOSERS.items()))
    return {"proposals": proposals}

In [5]:
def aggregate_proposals(state: MoAState) -> dict:
    """Synthesize by identifying where perspectives agree and disagree."""
    proposals_text = "\n\n---\n\n".join(state["proposals"])
    response = aggregator.invoke(
        f"Three experts assessed the same product launch from different angles.\n\n"
        f"{proposals_text}\n\n"
        f"Identify where the three perspectives AGREE (consensus), where they DISAGREE "
        f"(debate points), and synthesize a balanced assessment that incorporates the "
        f"strongest arguments from each.\n\n"
        f"Structure your response as:\n"
        f"1. CONSENSUS — Specific points where all three agree (quote or reference each)\n"
        f"2. DEBATE POINTS — Where they contradict each other, and who has the stronger argument\n"
        f"3. KEY RISKS — What does the skeptic flag that others miss?\n"
        f"4. KEY OPPORTUNITIES — What does the optimist see that others miss?\n"
        f"5. EXECUTION GAPS — What practical concerns does the pragmatist raise?\n"
        f"6. RECOMMENDATION — Go / No-Go / Go-with-conditions, and why."
    )
    return {"final_answer": response.content}

In [6]:
# Build graph
graph = StateGraph(MoAState)
graph.add_node("propose", generate_proposals)
graph.add_node("aggregate", aggregate_proposals)

graph.add_edge(START, "propose")
graph.add_edge("propose", "aggregate")
graph.add_edge("aggregate", END)

moa = graph.compile()

In [7]:
plot_mermaid(moa)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	propose(propose)
	aggregate(aggregate)
	__end__([<p>__end__</p>]):::last
	__start__ --> propose;
	propose --> aggregate;
	aggregate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
result = moa.invoke({
    "question": "We're launching an AI-powered medical billing assistant for small clinics. "
                "It auto-codes diagnoses and submits insurance claims. Assess this product launch."
})
print(result["final_answer"])

Certainly! Here's a structured analysis based on the perspectives provided:

1. **CONSENSUS**:
   - **Market Demand**: All three perspectives acknowledge the need for solutions that streamline operations in small clinics. The Optimist highlights the demand for reducing administrative burdens, while the Pragmatist includes this in their timeline and resource planning.
   - **Regulatory Compliance**: Both the Skeptic and Pragmatist emphasize the importance of adhering to healthcare regulations like HIPAA. The Optimist indirectly supports this by mentioning the product's ability to ensure compliance.
   - **Integration with Existing Systems**: The need for integration with EHR systems is noted by both the Skeptic and Pragmatist, indicating a shared understanding of its importance.

2. **DEBATE POINTS**:
   - **Data Privacy and Security**: The Skeptic raises concerns about data breaches and privacy, while the Optimist focuses on the product's potential to enhance accuracy and compliance. T

In [9]:
# Streaming

stream_invoke(
    moa, {
        "question": "We're launching an AI-powered medical billing assistant for small clinics. "
                    "It auto-codes diagnoses and submits insurance claims. Assess this product launch."
    }
)


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'question': "We're launching an AI-powered medical billing assistant for small clinics. It auto-codes diagnoses and submits insurance claims. Assess this product launch.",
 'proposals': ["[OPTIMIST]: Launching an AI-powered medical billing assistant for small clinics is a brilliant and timely initiative with immense potential for success. Here's why:\n\n1. **Market Demand**: The healthcare industry is increasingly looking for ways to streamline operations and reduce administrative burdens. Small clinics, in particular, often struggle with the complexities of medical billing due to limited resources. Your product addresses a critical pain point by automating the billing process, making it highly appealing to this target market.\n\n2. **Cost Efficiency**: Small clinics operate on tight budgets and are constantly seeking ways to cut costs. By automating the coding and claims submission process, your product can significantly reduce the need for extensive billing staff, leading to substan